# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates step-by-step loading, inspection, and exploration of the FAIR² dataset using the `mlcroissant` library, referencing each entity by its Croissant `@id`.

### Dataset Source

The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if missing
!pip install -U mlcroissant

## 1. Data Loading

Load dataset metadata and data records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a CroissantObject

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review the available record sets and their fields. All entities are referenced by their `@id`.

Let's enumerate all record sets, their IDs, and the fields within each record set.

In [ ]:
# Fetch all record sets from the dataset (by @id)
all_record_sets = dataset.record_sets

print(f"Found {len(all_record_sets)} record sets in the dataset.\n")

for rs in all_record_sets:
    print(f"Record set: {rs['@id']}")
    # Each record set contains fields by @id
    field_ids = [field['@id'] for field in rs.get('field', [])]
    print("  Fields (@id):")
    for fid in field_ids:
        print(f"    - {fid}")
    print("\n")

## 3. Data Extraction

Extract data from each record set into pandas DataFrames for further analysis. Use only the Croissant `@id` to reference record sets and fields.

All extracted DataFrames are keyed by the record set `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in all_record_sets]

# Extract data for each record set (@id)
dataframes = dict()
for rs_id in record_set_ids:
    # Each item yielded is a dict with Croissant @id keys for columns
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set {rs_id}")

# For demonstration, show columns for the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nColumns (@id) for first record set ({first_rs}):")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

We'll perform common data processing steps such as filtering, normalization, and grouping, referencing all fields *by their Croissant `@id`*.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` with values found in the previous section (get from first record set for demonstration).

In [ ]:
import numpy as np

# For demonstration, choose first record set and inspect columns
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    if not df.empty:
        print(f"Available columns (@id) in {rs_id}:")
        print(df.columns.tolist())

        # Try to auto-detect a numeric field (by basic dtype heuristic)
        numeric_field_id = None
        for col in df.columns:
            # Try to convert first 10 non-null entries to float
            series = pd.to_numeric(df[col], errors='coerce')
            if series.notnull().sum() > 0:
                numeric_field_id = col
                break
        if numeric_field_id is None:
            print("No numeric field detected.")
        else:
            print(f"Using '{numeric_field_id}' as numeric field for example analysis.")

        # Filter rows with numeric_field_id > threshold (e.g., mean)
        if numeric_field_id:
            series = pd.to_numeric(df[numeric_field_id], errors='coerce')
            mean_value = series.mean()
            filtered_df = df[series > mean_value].copy()
            print(f"Filtered records in {rs_id} where {numeric_field_id} > mean ({mean_value:.2f}): {len(filtered_df)} rows")
            display(filtered_df.head())

            # Normalize the numeric column
            filtered_df[f"{numeric_field_id}_normalized"] = (series[series > mean_value] - mean_value) / series.std()
            print("\nNormalized numeric field example:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a non-numeric column (auto)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id and numeric_field_id:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped.head())

## 5. Visualization

Visualize basic statistics of a chosen field, referenced by `@id`. For demonstration, we use the same numeric field from the previous section.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution if a numeric field is detected
if record_set_ids and numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouped statistics exist, show bar plot
    if 'grouped' in locals() and group_field_id is not None:
        plt.figure(figsize=(9,5))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² clinicopathological colorectal cancer survivor dataset with `mlcroissant` using only Croissant `@id` references.

- We identified all available record sets and fields by their unique `@id`s.
- Data from each record set is easily accessible as a pandas DataFrame with columns named by their `@id`.
- We demonstrated basic filtering, normalization, and grouping operations using field IDs, and visualized distributions of numeric attributes.

> This approach ensures referencing remains robust to evolving schema names and structure, and may be extended to more advanced analyses or visualizations as needed.